In [1]:
import duckdb
import pandas as pd

# подключаемся к базе
con = duckdb.connect('/Users/tatyanaagalakova/Desktop/vam_fashion/vam_fashion.duckdb')

# загружаем CSV
con.execute("""
    CREATE TABLE IF NOT EXISTS raw_vam_fashion AS 
    SELECT * FROM read_csv_auto('/Users/tatyanaagalakova/Desktop/vam_fashion.csv')
""")

# проверяем
result = con.execute("SELECT COUNT(*) FROM raw_vam_fashion").fetchone()
print(f"Записей в базе: {result[0]}")

Записей в базе: 1000


In [3]:
con.execute("""
    CREATE OR REPLACE VIEW stg_vam_fashion AS
    SELECT
        accessionNumber                          AS accession_number,
        accessionYear                            AS accession_year,
        systemNumber                             AS system_number,
        objectType                               AS object_type,
        _primaryTitle                            AS title,
        _primaryPlace                            AS place,
        _primaryMaker__name                      AS maker_name,
        _primaryMaker__association               AS maker_association,
        _primaryDate                             AS date_raw,
        _sampleMaterial                          AS material,
        _sampleTechnique                         AS technique,
        _sampleStyle                             AS style,
        _currentLocation__displayName            AS current_location
    FROM raw_vam_fashion
    WHERE accessionNumber IS NOT NULL
""")

result = con.execute("SELECT COUNT(*) FROM stg_vam_fashion").fetchone()
print(f"Staging: {result[0]} записей")

Staging: 1000 записей


In [9]:
result = con.execute("""
    SELECT place, COUNT(*) as count
    FROM stg_vam_fashion
    GROUP BY place
    ORDER BY count DESC
""").fetchdf()

print(result)

                 place  count
0               France    518
1        Great Britain    109
2              England     95
3               London     75
4              Britain     36
5                 None     33
6                Paris     27
7                 Rome     14
8                Italy     14
9            Nuremberg     13
10       United States      7
11              Vienna      4
12            New York      3
13               India      3
14              Mexico      3
15                  UK      3
16              Europe      3
17             Norwich      2
18                 USA      2
19              Hetian      2
20               Marne      2
21      Vicenza (town)      2
22               Japan      2
23           Greenwich      2
24             Paisley      2
25           Guangdong      2
26               Dhaka      1
27             Nigeria      1
28              Venice      1
29            Scotland      1
30               Spain      1
31  St Leonards-on-Sea      1
32        

In [11]:
# INTERMEDIATE — добавляем логику
con.execute("""
    CREATE OR REPLACE VIEW int_vam_fashion AS
    SELECT
        accession_number,
        accession_year,
        object_type,
        title,
        maker_name,
        material,
        technique,
        style,
        current_location,
        place,
        -- извлекаем примерный год из текстового поля
        CASE 
            WHEN date_raw LIKE '%1700%' OR date_raw LIKE '%17__' THEN '18th century'
            WHEN date_raw LIKE '%18__' OR date_raw LIKE '%1800%' THEN '19th century'
            WHEN date_raw LIKE '%19__' OR date_raw LIKE '%1900%' THEN '20th century'
            WHEN date_raw LIKE '%20__' OR date_raw LIKE '%2000%' THEN '21st century'
            ELSE 'Unknown'
        END AS century,
        -- группируем страны
        CASE
            WHEN place ILIKE '%france%' OR place ILIKE '%paris%' THEN 'France'
            WHEN place ILIKE '%britain%' OR place ILIKE '%england%' OR place ILIKE '%london%' THEN 'Great Britain'
            WHEN place ILIKE '%italy%' OR place ILIKE '%rome%' THEN 'Italy'
            WHEN place ILIKE '%united states%' OR place ILIKE '%new york%' THEN 'USA'
            WHEN place ILIKE '%united states%' OR place ILIKE '%new york%' THEN 'USA'
            ELSE COALESCE(place, 'Unknown')
        END AS country_grouped
    FROM stg_vam_fashion
""")

result = con.execute("SELECT COUNT(*) FROM int_vam_fashion").fetchone()
print(f"Intermediate: {result[0]} записей")

Intermediate: 1000 записей


In [13]:
# MARTS — финальные метрики
con.execute("""
    CREATE OR REPLACE TABLE mart_vam_fashion AS
    SELECT
        country_grouped,
        century,
        object_type,
        material,
        COUNT(*) as total_objects,
        COUNT(DISTINCT maker_name) as unique_makers,
        COUNT(DISTINCT technique) as unique_techniques
    FROM int_vam_fashion
    GROUP BY country_grouped, century, object_type, material
    ORDER BY total_objects DESC
""")

result = con.execute("SELECT * FROM mart_vam_fashion LIMIT 10").fetchdf()
print(result)

  country_grouped       century        object_type  \
0          France  20th century            Drawing   
1          France       Unknown            Drawing   
2   Great Britain  20th century             Design   
3          France  20th century     Fashion design   
4   Great Britain  20th century      Greeting card   
5   Great Britain  20th century              Print   
6          France  20th century     Fashion design   
7         Unknown  20th century              Print   
8       Nuremberg  18th century  Embroidery design   
9           Italy  19th century             Button   

                      material  total_objects  unique_makers  \
0                       pencil            217              1   
1                       pencil            211              1   
2                        paper             55             23   
3                       pencil             54              3   
4                         card             16              2   
5                    

In [31]:
# ТЕСТЫ
tests = {}

# 1. not_null — accession_number не пустой
tests['not_null_accession'] = con.execute("""
    SELECT COUNT(*) FROM stg_vam_fashion 
    WHERE accession_number IS NULL
""").fetchone()[0] == 0

# 2. unique — accession_number уникальный
tests['unique_accession'] = con.execute("""
    SELECT COUNT(*) FROM (
        SELECT accession_number 
        FROM stg_vam_fashion 
        GROUP BY accession_number 
        HAVING COUNT(*) > 1
    )
""").fetchone()[0] == 0

# 3. not_null — object_type не пустой
tests['not_null_object_type'] = con.execute("""
    SELECT COUNT(*) FROM stg_vam_fashion 
    WHERE object_type IS NULL
""").fetchone()[0] == 0

# 4. accepted_values — только известные страны
tests['accepted_countries'] = con.execute("""
    SELECT COUNT(*) FROM int_vam_fashion
    WHERE country_grouped NOT IN (
        'France', 'Great Britain', 'Italy', 'USA', 'India', 'Japan', 
        'China', 'Iran', 'Spain', 'Mexico', 'Jamaica', 'Nigeria', 
        'Egypt', 'Portugal', 'Netherlands', 'Austria', 'Poland', 
        'Czech Republic', 'Europe (general)', 'Native America', 
        'Germany', 'Cuba', 'Unknown'
    )
""").fetchone()[0] == 0

status = '✅ PASS' if tests['accepted_countries'] else '❌ FAIL'
print(f"{status} — accepted_countries")

# 5. not_null — century не пустой
tests['not_null_century'] = con.execute("""
    SELECT COUNT(*) FROM int_vam_fashion
    WHERE century IS NULL
""").fetchone()[0] == 0

for test_name, passed in tests.items():
    status = '✅ PASS' if passed else '❌ FAIL'
    print(f"{status} — {test_name}")

✅ PASS — accepted_countries
✅ PASS — not_null_accession
✅ PASS — unique_accession
✅ PASS — not_null_object_type
✅ PASS — accepted_countries
✅ PASS — not_null_century


In [17]:
result = con.execute("""
    SELECT DISTINCT country_grouped, COUNT(*) as count
    FROM int_vam_fashion
    WHERE country_grouped NOT IN ('France', 'Great Britain', 'Italy', 'USA', 'Unknown', 'Nuremberg')
    GROUP BY country_grouped
    ORDER BY count DESC
""").fetchdf()

print(result)

       country_grouped  count
0               Vienna      4
1                India      3
2               Mexico      3
3                   UK      3
4               Europe      3
5            Greenwich      2
6       Vicenza (town)      2
7               Hetian      2
8              Norwich      2
9                Japan      2
10               Marne      2
11           Guangdong      2
12             Paisley      2
13            Biarritz      1
14            Varanasi      1
15             Nigeria      1
16              Kraków      1
17           Amsterdam      1
18              Dieppe      1
19             Gujarat      1
20              Havana      1
21               Egypt      1
22           Edinburgh      1
23                Iran      1
24               Dhaka      1
25  St Leonards-on-Sea      1
26      Czech Republic      1
27            Portugal      1
28              Venice      1
29             Jamaica      1
30            Scotland      1
31               Lyons      1
32   Easte

In [25]:
con.execute("""
    CREATE OR REPLACE VIEW int_vam_fashion AS
    SELECT
        accession_number,
        accession_year,
        object_type,
        title,
        maker_name,
        material,
        technique,
        style,
        current_location,
        place,
        CASE 
            WHEN date_raw LIKE '%17__%' THEN '18th century'
            WHEN date_raw LIKE '%18__%' THEN '19th century'
            WHEN date_raw LIKE '%19__%' THEN '20th century'
            WHEN date_raw LIKE '%20__%' THEN '21st century'
            ELSE 'Unknown'
        END AS century,
        CASE
            WHEN place ILIKE '%france%' OR place ILIKE '%paris%' OR place ILIKE '%lyon%' OR place ILIKE '%biarritz%' OR place ILIKE '%dieppe%' OR place ILIKE '%marne%' THEN 'France'
            WHEN place ILIKE '%britain%' OR place ILIKE '%england%' OR place ILIKE '%london%' OR place ILIKE '%uk%' OR place ILIKE '%greenwich%' OR place ILIKE '%norwich%' OR place ILIKE '%paisley%' OR place ILIKE '%scotland%' OR place ILIKE '%edinburgh%' OR place ILIKE '%st leonards%' THEN 'Great Britain'
            WHEN place ILIKE '%italy%' OR place ILIKE '%rome%' OR place ILIKE '%venice%' OR place ILIKE '%vicenza%' THEN 'Italy'
            WHEN place ILIKE '%united states%' OR place ILIKE '%new york%' THEN 'USA'
            WHEN place ILIKE '%india%' OR place ILIKE '%gujarat%' OR place ILIKE '%varanasi%' OR place ILIKE '%dhaka%' OR place ILIKE '%deccan%' THEN 'India'
            WHEN place ILIKE '%japan%' THEN 'Japan'
            WHEN place ILIKE '%china%' OR place ILIKE '%guangdong%' OR place ILIKE '%hetian%' THEN 'China'
            WHEN place ILIKE '%iran%' THEN 'Iran'
            WHEN place ILIKE '%spain%' THEN 'Spain'
            WHEN place ILIKE '%mexico%' THEN 'Mexico'
            WHEN place ILIKE '%jamaica%' THEN 'Jamaica'
            WHEN place ILIKE '%nigeria%' THEN 'Nigeria'
            WHEN place ILIKE '%egypt%' THEN 'Egypt'
            WHEN place ILIKE '%portugal%' THEN 'Portugal'
            WHEN place ILIKE '%amsterdam%' THEN 'Netherlands'
            WHEN place ILIKE '%vienna%' THEN 'Austria'
            WHEN place ILIKE '%krak%' THEN 'Poland'
            WHEN place ILIKE '%czech%' THEN 'Czech Republic'
            WHEN place ILIKE '%europe%' THEN 'Europe (general)'
            WHEN place ILIKE '%eastern woodlands%' THEN 'Native America'
            WHEN place ILIKE '%nuremberg%' THEN 'Germany'
            WHEN place ILIKE '%havana%' THEN 'Cuba'
            ELSE COALESCE(place, 'Unknown')
        END AS country_grouped
    FROM stg_vam_fashion
""")

print("Обновлено!")

Обновлено!


In [29]:
result = con.execute("""
    SELECT DISTINCT country_grouped, COUNT(*) as count
    FROM int_vam_fashion
    WHERE country_grouped NOT IN ('France', 'Great Britain', 'Italy', 'USA', 'India', 'Japan', 'China', 'Iran', 'Spain', 'Mexico', 'Jamaica', 'Nigeria', 'Egypt', 'Portugal', 'Netherlands', 'Austria', 'Poland', 'Czech Republic', 'Europe (general)', 'Native America', 'Germany', 'Cuba', 'Unknown')
    GROUP BY country_grouped
    ORDER BY count DESC
""").fetchdf()

print(result)

Empty DataFrame
Columns: [country_grouped, count]
Index: []
